# Ledger retention & cohort analysis
Run top to bottom (Restart → Run All). `ledger_loader.py` must be in the same folder as this notebook.

**Cohorts:** sections 5–8  ·  **Inactive clients & chats:** sections 9–11

## 1. Setup

In [ ]:
from ledger_loader import get_final_ledger
import pandas as pd
import numpy as np
import os
import glob
import re
import json
import matplotlib as mpl
from IPython.display import display
from openpyxl.cell.rich_text import CellRichText, TextBlock
from openpyxl.cell.text import InlineFont
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

CLIENTS_DIR      = r"C:\Users\Administrator\Videos\Clients"
IB_DIR           = r"C:\Users\Administrator\Videos\IB"
CHAT_DIR         = r"C:\Users\Administrator\Videos\Chat"
NOT_RETURNED_DIR = r"C:\Users\Administrator\Videos\Not Returned"
COHORT_DIR       = r"C:\Users\Administrator\Videos\co"


def latest_csv(folder, pattern="*.csv"):
    """Newest matching CSV in a folder, so a fresh download is always the one used."""
    files = glob.glob(os.path.join(folder, pattern))
    if not files:
        raise FileNotFoundError(f"No {pattern} files in {folder}")
    path = max(files, key=os.path.getmtime)
    print("Using:", path)
    return path

## 2. Load ledger and build monthly activity

In [ ]:
ledger = get_final_ledger()

entry = ["BILL"]
led = ledger[ledger["LedgerType"].isin(entry)].copy()
led["Month"] = led["CreatedAt"].dt.to_period("M")

activity = led[["ClientID", "Month"]].drop_duplicates()
activity["Cohort"] = activity.groupby("ClientID")["Month"].transform("min")
activity["MonthIdx"] = (activity["Month"] - activity["Cohort"]).apply(lambda x: x.n)

this_month = pd.Timestamp.today().to_period("M")   # or activity["Month"].max() for the latest month in the data
prev_month = this_month - 1
last_month = activity["Month"].max()               # end month for blanking future cohort cells
max_idx = activity["MonthIdx"].max()               # same M-columns on every cohort table

print("Ledger runs to:", ledger["CreatedAt"].max())

## 3. Lead source, migration status and IB referrals
Assumes `ARC Client ID` is a column in the clients file. If it comes from a different file, merge that file into `activity_source` here instead, before the `Client Status` line.

In [ ]:
clients = pd.read_csv(latest_csv(CLIENTS_DIR))
source_col = "Source One"

Leads_Map = clients[["Client ID", source_col, "ARC Client ID"]]
activity_source = activity.merge(Leads_Map, left_on="ClientID", right_on="Client ID", how="left")

activity_source[source_col] = activity_source[source_col].fillna("Unknown")
activity_source["Client Status"] = np.select(
    [activity_source["Client ID"].isna(), activity_source["ARC Client ID"].notna()],
    ["Unmatched", "Migrated"],
    default="Fresh",
)
activity_source.head()

In [ ]:
# sanity checks - look at these before building cohorts
print("Duplicate Client IDs in Leads_Map:", Leads_Map["Client ID"].duplicated().sum())          # should be 0
print("Clients not found in clients file:", activity_source.loc[activity_source["Client ID"].isna(), "ClientID"].nunique())
print("Clients with more than one status:", (activity_source.groupby("ClientID")["Client Status"].nunique() > 1).sum())  # should be 0

In [ ]:
IB = pd.read_csv(latest_csv(IB_DIR))

IB = IB[
    [
        "Broker Client ID",
        "Broker Referral Count",
        "Referred User Client ID",
        "Referred User Created At",
    ]
].copy()

IB["Referred User Created At"] = pd.to_datetime(
    IB["Referred User Created At"], format="%Y-%m-%d %H:%M:%S", errors="coerce"
)

IB_Child = IB[["Referred User Client ID"]]
IB.head()

## 4. Helper functions
All reusable functions live in this one cell.

In [ ]:
cmap = mpl.colormaps["Blues"].copy()
cmap.set_bad("white")
STATUS_ORDER = ["Migrated", "Fresh", "Unmatched"]      # order of the small breakdown line


def build_cohort(act):
    """Plain cohort table (used for the retention summaries)."""
    counts = (act.groupby(["Cohort", "MonthIdx"])["ClientID"].nunique()
                 .unstack(fill_value=0)
                 .reindex(columns=range(max_idx + 1), fill_value=0))
    size = counts[0]
    pct = counts.div(size, axis=0) * 100
    for cohort in pct.index:
        pct.loc[cohort, pct.columns > (last_month - cohort).n] = np.nan
    active = counts.where(pct.notna())

    # all cohorts combined: clients still active / clients whose cohort has reached that month
    overall = active.sum() / pct.notna().mul(size, axis=0).sum() * 100

    table = (pct.map("{:.0f}%".format) + " (" + active.map("{:,.0f}".format) + ")").where(pct.notna(), "—")
    table.columns = [f"M{c}" for c in table.columns]
    table.insert(0, "Size", size)
    table.index = table.index.strftime("%b %Y")
    table.index.name = "Cohort"

    styled = table.style.background_gradient(
        cmap=cmap, subset=table.columns[1:], gmap=pct.to_numpy(), axis=None, vmin=0, vmax=100)
    return styled, overall


def split_cohort(act, split_col="Client Status"):
    """Cohort where each cell is '712 (100%)' plus a small 'Migrated: 25 · Fresh: 687' line."""
    counts = (act.groupby(["Cohort", "MonthIdx"])["ClientID"].nunique()
                 .unstack(fill_value=0)
                 .reindex(columns=range(max_idx + 1), fill_value=0))
    size = counts[0]
    pct = counts.div(size, axis=0) * 100
    for cohort in pct.index:
        pct.loc[cohort, pct.columns > (last_month - cohort).n] = np.nan

    split = act.groupby(["Cohort", "MonthIdx", split_col])["ClientID"].nunique()

    main = pd.DataFrame("", index=pct.index, columns=pct.columns)
    sub = main.copy()
    for (cohort, idx), p in pct.stack().dropna().items():
        main.loc[cohort, idx] = f"{counts.loc[cohort, idx]:,} ({p:.0f}%)"
        parts = [f"{s}: {split.get((cohort, idx, s), 0):,}" for s in STATUS_ORDER
                 if s != "Unmatched" or split.get((cohort, idx, s), 0)]
        sub.loc[cohort, idx] = " · ".join(parts)
    return {"main": main, "sub": sub, "pct": pct, "size": size}


def show_split(grid, caption=None):
    """Notebook view of a split cohort: big line + small breakdown line."""
    cells = (grid["main"] + '<br><span style="font-size:0.72em;opacity:0.85">'
             + grid["sub"] + "</span>").where(grid["pct"].notna(), "—")
    cells.columns = [f"M{c}" for c in cells.columns]
    cells.insert(0, "Size", grid["size"])
    cells.index = cells.index.strftime("%b %Y")
    cells.index.name = "Cohort"
    sty = (cells.style
           .background_gradient(cmap=cmap, subset=cells.columns[1:], gmap=grid["pct"].to_numpy(),
                                axis=None, vmin=0, vmax=100)
           .set_properties(subset=cells.columns[1:], **{"text-align": "center", "white-space": "nowrap"}))
    return sty.set_caption(caption) if caption else sty


def _is_dark(rgb):
    lin = [c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4 for c in rgb]
    return 0.2126 * lin[0] + 0.7152 * lin[1] + 0.0722 * lin[2] < 0.408


def write_split_sheet(xw, name, grid):
    """Excel view of a split cohort: big line + small line in the same cell."""
    ws = xw.book.create_sheet(name)
    main, sub, pct, size = grid["main"], grid["sub"], grid["pct"], grid["size"]

    headers = ["Cohort", "Size"] + [f"M{c}" for c in pct.columns]
    for j, h in enumerate(headers, 1):
        cell = ws.cell(1, j, h)
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal="center")

    for i, cohort in enumerate(pct.index, 2):
        label_cell = ws.cell(i, 1, cohort.strftime("%b %Y"))
        label_cell.font = Font(bold=True)
        label_cell.alignment = Alignment(vertical="center")
        ws.cell(i, 2, int(size[cohort])).alignment = Alignment(horizontal="center", vertical="center")
        for j, idx in enumerate(pct.columns, 3):
            cell = ws.cell(i, j)
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            p = pct.loc[cohort, idx]
            if pd.isna(p):
                cell.value = "—"
                continue
            rgb = cmap(p / 100)[:3]
            txt = "FFFFFF" if _is_dark(rgb) else "000000"
            cell.fill = PatternFill("solid", fgColor=mpl.colors.rgb2hex(rgb)[1:].upper())
            cell.value = CellRichText(
                TextBlock(InlineFont(rFont="Calibri", sz=11, b=True, color=txt), main.loc[cohort, idx] + "\n"),
                TextBlock(InlineFont(rFont="Calibri", sz=8, color=txt), sub.loc[cohort, idx]),
            )
        ws.row_dimensions[i].height = 32

    ws.column_dimensions["A"].width = 11
    ws.column_dimensions["B"].width = 8
    for j in range(3, len(headers) + 1):
        ws.column_dimensions[get_column_letter(j)].width = 24
    ws.freeze_panes = "C2"


def label(key):
    return " - ".join(map(str, key)) if isinstance(key, tuple) else str(key)


def cohorts_by(act, cols):
    tables, overall, clients = {}, {}, {}
    for key, grp in act.groupby(cols):
        key = key[0] if len(cols) == 1 else key
        styled, overall[key] = build_cohort(grp)
        clients[key] = grp["ClientID"].nunique()
        tables[key] = styled.set_caption(f"{label(key)}: {clients[key]:,} clients")

    summary = pd.DataFrame.from_dict(overall, orient="index").round(0)
    summary.columns = [f"M{c}" for c in summary.columns]
    summary.insert(0, "Clients", pd.Series(clients))
    summary.index.names = cols
    summary = summary.sort_values("Clients", ascending=False) if len(cols) == 1 else summary.sort_index()
    return tables, summary


def export_cohorts(act, file_path, all_label="All Clients", extra_sheets=None):
    """Summary sheets + split cohort for everyone + one split cohort per source, in one workbook."""
    _, src_sum = cohorts_by(act, [source_col])
    _, status_sum = cohorts_by(act, ["Client Status"])
    _, combo_sum = cohorts_by(act, [source_col, "Client Status"])
    grids = {src: split_cohort(grp) for src, grp in act.groupby(source_col)}

    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    used = set()
    with pd.ExcelWriter(file_path, engine="openpyxl") as xw:
        for name, df in (extra_sheets or {}).items():
            df.to_excel(xw, sheet_name=sheet_name(name, used))
        src_sum.to_excel(xw, sheet_name=sheet_name("Summary - Source", used))
        status_sum.to_excel(xw, sheet_name=sheet_name("Summary - Status", used))
        combo_sum.to_excel(xw, sheet_name=sheet_name("Summary - Source x Status", used))
        write_split_sheet(xw, sheet_name(all_label, used), split_cohort(act))
        for src in src_sum.index:                    # biggest sources first
            write_split_sheet(xw, sheet_name(src, used), grids[src])
    print("Saved:", file_path)


def money_summary(ledger, client_ids):
    sub = ledger[ledger["ClientID"].isin(client_ids)]
    lt = sub["LedgerType"].str.strip().str.upper()
    money = (
        pd.DataFrame({
            "ClientID": sub["ClientID"],
            "total_deposit": sub["Credit"].where(lt == "DEPOSIT", 0),
            "total_withdraw": sub["Debit"].where(lt == "WITHDRAW", 0)
                            - sub["Credit"].where(lt == "WITHDRAW CANCELLED", 0),
        })
        .groupby("ClientID")[["total_deposit", "total_withdraw"]]
        .sum()
    )
    money["net_deposit"] = money["total_deposit"] - money["total_withdraw"]
    money["last_balance"] = sub.sort_values("CreatedAt").groupby("ClientID")["Balance"].last()
    return money


def sheet_name(name, used):
    base = re.sub(r"[\[\]:*?/\\]", "_", str(name)).strip("' ")[:31] or "Blank"
    name, i = base, 2
    while name.lower() in used:                      # Excel sheet names must be unique, ignoring case
        name = f"{base[:28]}_{i}"
        i += 1
    used.add(name.lower())
    return name


def norm_id(s):
    """Trim and upper-case client IDs so the same client matches across files."""
    return s.astype(str).str.strip().str.upper()

## 5. Overall cohort with Migrated / Fresh breakdown

In [ ]:
all_grid = split_cohort(activity_source)
show_split(all_grid)

## 6. Cohorts by source and retention summaries

In [ ]:
_, source_summary = cohorts_by(activity_source, [source_col])
_, status_summary = cohorts_by(activity_source, ["Client Status"])
_, combo_summary = cohorts_by(activity_source, [source_col, "Client Status"])

display(source_summary, status_summary, combo_summary)

In [ ]:
# view any one source's cohort, e.g. the biggest one
top_source = source_summary.index[0]
show_split(split_cohort(activity_source[activity_source[source_col] == top_source]), caption=top_source)

## 7. Export all cohorts to one workbook

In [ ]:
export_cohorts(activity_source, os.path.join(COHORT_DIR, f"cohorts_{last_month}.xlsx"))

## 8. IB-referred clients

In [ ]:
ib_ids = norm_id(IB_Child["Referred User Client ID"].dropna()).unique()

is_ib = norm_id(activity_source["ClientID"]).isin(ib_ids)
ib_activity = activity_source[is_ib].copy()

print(f"Referred users in IB file:          {len(ib_ids):,}")
print(f"Found in activity (billed 1+ time): {ib_activity['ClientID'].nunique():,}")
print(f"Never billed:                       {len(ib_ids) - ib_activity['ClientID'].nunique():,}")

_, ib_vs_others = cohorts_by(
    activity_source.assign(Referral=np.where(is_ib, "IB referred", "Not IB referred")),
    ["Referral"],
)
ib_grid = split_cohort(ib_activity)

display(ib_vs_others)
show_split(ib_grid, caption=f"IB-referred clients: {ib_activity['ClientID'].nunique():,}")

In [ ]:
export_cohorts(
    ib_activity,
    os.path.join(COHORT_DIR, f"ib_cohorts_{last_month}.xlsx"),
    all_label="All IB Clients",
    extra_sheets={"IB vs Others": ib_vs_others},
)
os.startfile(COHORT_DIR)

## 9. Clients who haven't billed this month

In [ ]:
prev_ids = activity.loc[activity["Month"] == prev_month, "ClientID"]
this_ids = activity.loc[activity["Month"] == this_month, "ClientID"]
missing = prev_ids[~prev_ids.isin(this_ids)]

not_returned = (
    led[led["ClientID"].isin(missing)]
    .assign(in_prev=lambda d: d["Month"] == prev_month)
    .groupby("ClientID")
    .agg(cohort=("Month", "min"),
         bills_prev_month=("in_prev", "sum"),
         last_bill=("CreatedAt", "max"))
    .join(money_summary(ledger, missing))
    .sort_values("bills_prev_month", ascending=False)
)
print(f"{len(not_returned)} of {len(prev_ids)} {prev_month.strftime('%b %Y')} traders "
      f"have no BILL in {this_month.strftime('%b %Y')}")
not_returned

In [ ]:
inactive = (
    activity[(activity["Month"] < this_month) & ~activity["ClientID"].isin(this_ids)]
    .groupby("ClientID")
    .agg(cohort=("Month", "min"),
         last_active_month=("Month", "max"),
         active_months=("Month", "nunique"))
)
inactive["months_since_last_bill"] = (this_month - inactive["last_active_month"]).apply(lambda x: x.n)
inactive["last_bill"] = led.groupby("ClientID")["CreatedAt"].max()
inactive = (inactive.join(money_summary(ledger, inactive.index))
            .sort_values(["months_since_last_bill", "last_balance"], ascending=[True, False]))

ladder = inactive.groupby("months_since_last_bill").agg(
    clients=("cohort", "size"),
    net_deposit=("net_deposit", "sum"),
    balance_left=("last_balance", "sum"),
)
ladder

## 10. Chat history for clients who haven't returned

In [ ]:
chat = pd.read_csv(latest_csv(CHAT_DIR, "History*.csv"), skiprows=2, header=0)


def parse_json(x):
    try:
        return json.loads(x) if isinstance(x, str) else {}
    except ValueError:
        return {}


details_df = pd.concat(
    [
        chat[["Created Time", "Tags"]].reset_index(drop=True),
        pd.json_normalize(chat["Customer Info"].apply(parse_json)).reset_index(drop=True),
    ],
    axis=1,
)[["Created Time", "Tags", "Client_Id"]]

details_df = details_df.dropna(subset=["Tags", "Client_Id"])
details_df = details_df[details_df["Tags"].astype(str).str.strip() != ""].copy()
details_df["Client_Id"] = norm_id(details_df["Client_Id"])
details_df["Created Time"] = pd.to_datetime(details_df["Created Time"], dayfirst=True, errors="coerce")
details_df = details_df.sort_values("Created Time", ascending=False)      # newest chat first

details_df["Chat Details"] = (
    details_df["Created Time"].dt.strftime("%d %b %Y, %H:%M").fillna("Unknown time")
    + " — " + details_df["Tags"].astype(str).str.strip()
)
chat_lookup = details_df.groupby("Client_Id")["Chat Details"].agg("\n\n".join)

not_returned["Chat Details"] = (
    norm_id(not_returned.index.to_series()).map(chat_lookup).fillna("No matching chat")
)
print("Not-returned clients with a chat:", (not_returned["Chat Details"] != "No matching chat").sum())
not_returned

## 11. Export inactive clients workbook

In [ ]:
os.makedirs(NOT_RETURNED_DIR, exist_ok=True)
file_path = os.path.join(NOT_RETURNED_DIR, f"inactive_clients_{this_month}.xlsx")

with pd.ExcelWriter(file_path, engine="openpyxl") as xw:
    not_returned.astype({"cohort": str}).to_excel(xw, sheet_name="Not returned last month")
    inactive.astype({"cohort": str, "last_active_month": str}).to_excel(xw, sheet_name="All inactive")
    ladder.to_excel(xw, sheet_name="Summary")

    # make the multi-line Chat Details column readable
    ws = xw.sheets["Not returned last month"]
    col = get_column_letter(not_returned.columns.get_loc("Chat Details") + 2)   # +2: 1-based and index column
    ws.column_dimensions[col].width = 60
    for cell in ws[col][1:]:
        cell.alignment = Alignment(wrap_text=True, vertical="top")

print("Saved:", file_path)
os.startfile(NOT_RETURNED_DIR)